In [ ]:
import numpy as np
import os
import pandas as pd
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from collections import defaultdict

def cohen_d_calculation(model, target, variable, data):
    n = data.shape[0]
    beta = model.params.loc[variable]
    y = data[target] - beta * data[variable]
    sse = np.sum((y - np.mean(y))**2)
    sd_pooled = np.sqrt(sse / (n - 2))
    cohen_d = beta / sd_pooled
    return cohen_d
    
header = list(range(0, 5, 1)) 
df = pd.read_csv(r'D:\cohort1.csv', index_col=0, header=header)
columns = df.columns.to_frame(index=False)
columns[columns.columns[1:]] = columns[columns.columns[1:]].astype(float)
df.columns = pd.MultiIndex.from_frame(columns, names=df.columns.names)
coef = defaultdict(dict)
prob = defaultdict(dict)
eta = defaultdict(dict)
cohen_ds = defaultdict(dict)

for gene in df.index[:]:
    data = df.loc[gene].dropna().reset_index()
    columns = data.columns.tolist()
    columns[columns.index(gene)] = 'gene'
    data.columns = columns
    model = ols("gene ~ " + ' + '.join(list(df.columns.names)[1:]), data=data).fit()
    coef[gene] = model.params.to_dict()
    prob[gene] = model.pvalues.to_dict()
    variables = np.hstack(['Intercept', columns[1:-1]])
    cohen_ds_dict = {}
    for i, variable in enumerate(variables[1:]):
        cohen_ds_dict[variable] = cohen_d_calculation(model, 'gene', variable, data)
    cohen_ds[gene] = cohen_ds_dict
    try:
        output = anova_lm(model, typ=2)
        output['F'] = output['F']/output['F'].sum()
        eta[gene] = output['F'].iloc[:-1].to_dict()
    except:
        print(gene)

out = pd.concat([pd.DataFrame().from_dict(coef).rename(lambda x: x+' coef'), pd.DataFrame().from_dict(prob).rename(lambda x: x+' prob'), pd.DataFrame().from_dict(eta).rename(lambda x: x+' eta'), pd.DataFrame().from_dict(cohen_ds).rename(lambda x: x+' effect size')]).sort_index().T
out.to_csv(r'D:\cohort1-A.csv')